# Are Sisters, or Cousins, Morphologically More Alike Than Random? (vem 1uM, 144hrs Pre-Treatment)

`vem_1uM_144hrsbefore` imaging starts 6 days before 1uM vemurafenib is
added (frame 7 is day 0; frame 6 - the day before, day -1 - is
currently missing entirely from this dataset, and falls outside both
named ranges below). Frames 1-5 are therefore **before-drug** (effectively
naive growth); frames 7-11 are **after-drug**. Both ranges' end frames
are notebook-level parameters (`config.BEFORE_DRUG_END_FRAME`,
`config.AFTER_DRUG_END_FRAME`). A third, wholly separate **naive**
condition (never exposed to any drug at all, its own independent CAJAL
run) is included as a sanity-check control over the same number of
frames as the before-drug window - it should look like before-drug,
since neither has seen a drug yet.

**Question**: is a cell's morphology more similar to its sister's (born
from the same division) than to a random cell's? What about its
cousins' (its parent and its parent's sister's children - i.e. sharing
a grandparent, not a parent)? This notebook answers both, separately
for before-drug, after-drug, and naive, so all three can be compared:
does vemurafenib change how (dis)similar relatives are, relative to how
things looked before treatment (and relative to cells that were never
treated at all)?

Rather than collapsing each relationship's real pairs down to a single
mean distance and comparing THAT against a null of means (which hides
whether the effect holds for most real pairs or is driven by just a
few), this compares the FULL distribution of real pairs' own distances
against a null distribution of individual random-pair distances, via a
one-sided Mann-Whitney U test.

Sister/cousin relationships are reconstructed directly from each
condition's own full lineage tracking (`node_id`/`pred_node_id`/`lin_id` -
see `shared/lineage_tree.py`), **not** from `lineage_frame7_to_frame1.mat`
(that file only resolves the single frame-1 ancestor of each day-0
lineage, collapsing every division in between - the generation-by-
generation structure used here needs more than that; naive doesn't even
have such a file, since it has no day-0 drug-addition event to anchor one).

Requires `main.py` to have already been run for BOTH conditions (for
each condition's own `selected_metadata.csv` and
`{cell,nuc}_qgw_075P_dists.csv` under its own
`output/{condition}/cajal_output/`).

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/04_vem_144hr_lineage_relatedness/notebooks/), so the analysis
# dir - and config.py - is one level up, and the repo root - where
# shared/ lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR is relative to the analysis directory, not this
# notebook's own cwd - re-anchor it to an absolute path once, here.
# config.cajal_output_dir()/output_dir() both derive from this one
# variable, so re-anchoring it here is enough for every condition's own
# subtree too.
config.OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)

from shared.cajal.embed import load_distance_matrix
from shared.load_data import parse_label
from shared.lineage_tree import (
    build_segments, find_sister_pairs, find_cousin_pairs,
    observations_in_range, real_index_pairs, pool_in_range,
)
from shared.pair_similarity import pair_distance_permutation_test, plot_pair_similarity_test

## Method

1. **Reconstruct every lineage tree**, once per condition (`vem` in
   frames 1-`config.AFTER_DRUG_END_FRAME`, `naive` in frames
   1-`config.BEFORE_DRUG_END_FRAME`) from each condition's FULL metadata
   (not just the cells CAJAL actually ran on - see below), and find
   every sister pair and cousin pair in it (`shared/lineage_tree.py`).
2. **For each modality** (cell, nucleus) **and each group**
   (before-drug, after-drug, naive - the first two share `vem`'s one
   lineage tree/CAJAL run, just restricted to a different frame range;
   naive has its own entirely separate tree/CAJAL run):
   - Restrict each pair's two sides to whichever of their own frames
     fall in this group's range AND actually got a CAJAL embedding
     (survived `main.py`'s subsampling - a simple, lineage-unaware
     subsample, so which true pairs keep both members is left to
     chance). Use every surviving (frame, pixel_id) combination on each
     side as its own real pair - sisters and cousins alike are **not**
     required to share a frame (a sister's own track can run longer
     than its sibling's; a cousin pair's parents can divide at
     different times).
   - Build ONE shared null distribution: many individual random pairs,
     drawn (with replacement) from every CAJAL-embedded cell in this
     same group, any frame - not frame-matched, same as the real pairs
     above.
   - Compare the real sister pairs' own distances, and separately the
     real cousin pairs' own distances, against that SAME null, via a
     one-sided Mann-Whitney U test (`shared/pair_similarity.py`'s
     `pair_distance_permutation_test` - called with the same pool/seed
     both times, which is what makes the null identical for both).
3. **Plot**: one figure per (modality, group) - the null distribution of
   individual random-pair distances, overlaid with the real sister-pair
   and cousin-pair distance distributions (`plot_pair_similarity_test`).

In [ ]:
PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '01_relatedness')
SVG_DIR = os.path.join(PLOTS_DIR, 'svgs')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}
os.makedirs(SVG_DIR, exist_ok=True)

# group -> (condition, (start, end) frame, inclusive) - before-drug and
# after-drug split vem's ONE lineage tree/CAJAL run into two
# sub-ranges; naive is a wholly separate condition (own tree, own CAJAL
# run), used as a sanity-check control - expected to look like
# before-drug, since neither has seen a drug yet
GROUPS = {
    'before-drug': ('vem', (1, config.BEFORE_DRUG_END_FRAME)),
    'after-drug':  ('vem', (config.AFTER_DRUG_START_FRAME, config.AFTER_DRUG_END_FRAME)),
    'naive':       ('naive', (1, config.CONDITIONS['naive']['max_frame'])),
}
CONDITIONS_USED = sorted({condition for condition, _ in GROUPS.values()})


def process_modality(modality, segments_by_condition):
    '''
    For one modality: load each condition's real distance matrix once,
    restrict the sister/cousin pairs found in that condition's own
    lineage tree to each of its groups' frame ranges in turn, and
    run+plot the test for each.

    Inputs:
        modality:              'cell' or 'nuc'
        segments_by_condition: {condition: segments from build_segments}
    '''
    dist_matrices = {}
    for condition in CONDITIONS_USED:
        dist_csv = os.path.join(config.cajal_output_dir(condition), f'{modality}_qgw_{config.N_POINTS:03d}P_dists.csv')
        labels, dist_matrix = load_distance_matrix(dist_csv)
        label_to_index = {}
        for i, label in enumerate(labels):
            _, frame, pixel_id = parse_label(label)
            label_to_index[(frame, pixel_id)] = i
        dist_matrices[condition] = (dist_matrix, label_to_index)

    for group_name, (condition, frame_range) in GROUPS.items():
        segments = segments_by_condition[condition]
        sister_pairs = find_sister_pairs(segments)
        cousin_pairs = find_cousin_pairs(segments)

        dist_matrix, label_to_index = dist_matrices[condition]
        real_sister_idx = real_index_pairs(sister_pairs, frame_range, label_to_index)
        real_cousin_idx = real_index_pairs(cousin_pairs, frame_range, label_to_index)
        pool = pool_in_range(label_to_index, frame_range)

        print(f"  [{group_name} ({condition}), frames {frame_range[0]}-{frame_range[1]}] "
              f"{len(real_sister_idx)} sister-pair observation(s), {len(real_cousin_idx)} cousin-pair observation(s), "
              f"{len(pool)} cell(s) in the pool")

        if len(real_sister_idx) == 0 or len(real_cousin_idx) == 0:
            print(f"    Skipping: need at least one sister-pair AND one cousin-pair observation with both sides sampled")
            continue

        # both calls share this SAME pool, n_null_samples, and seed,
        # which is what makes their null distributions identical (see
        # pair_distance_permutation_test's own docstring)
        sister_distances, null_distances, sister_p = pair_distance_permutation_test(
            dist_matrix, pool, real_sister_idx, n_null_samples=config.N_NULL_SAMPLES, seed=config.TEST_SEED,
        )
        cousin_distances, null_distances_check, cousin_p = pair_distance_permutation_test(
            dist_matrix, pool, real_cousin_idx, n_null_samples=config.N_NULL_SAMPLES, seed=config.TEST_SEED,
        )
        assert np.array_equal(null_distances, null_distances_check), "the two calls' null distributions must be identical"

        print(f"    sisters: mean distance = {sister_distances.mean():.4f} (Mann-Whitney U p = {sister_p:.4g})")
        print(f"    cousins: mean distance = {cousin_distances.mean():.4f} (Mann-Whitney U p = {cousin_p:.4g})")

        fig = plot_pair_similarity_test(null_distances, {
            'sisters': (sister_distances, sister_p),
            'cousins': (cousin_distances, cousin_p),
        }, title=f'{MODALITY_NAMES[modality]} morphology: {group_name} (frames {frame_range[0]}-{frame_range[1]})')
        fig.savefig(os.path.join(PLOTS_DIR, f'{modality}_{group_name}.png'), dpi=200, bbox_inches='tight')
        fig.savefig(os.path.join(SVG_DIR, f'{modality}_{group_name}.svg'), bbox_inches='tight')
        display(fig)   # show it in the notebook's own output, not just save it to disk
        plt.close(fig)
        print(f"    Saved to {PLOTS_DIR}")

        yield {
            'modality': modality, 'group': group_name,
            'sister_distance': sister_distances, 'sister_p_value': sister_p,
            'cousin_distance': cousin_distances, 'cousin_p_value': cousin_p,
            'null_distance': null_distances,
        }

## Build each condition's lineage tree (once - shared across both modalities)

In [ ]:
segments_by_condition = {}
for condition in CONDITIONS_USED:
    max_frame = config.CONDITIONS[condition]['max_frame']
    full_metadata = pd.read_csv(config.metadata_csv_path(condition))

    # tree-building uses the FULL metadata (every tracked cell, has_mask
    # True or False) - has_mask only reflects whether a usable mask crop
    # could be extracted, not whether the cell/lineage tracking itself
    # exists. Filtering to has_mask=True here would drop intermediate
    # rows a later frame's pred_node_id still needs, breaking chains
    # that are otherwise intact.
    full_metadata = full_metadata[full_metadata['frame'] <= max_frame].reset_index(drop=True)

    segments_by_condition[condition] = build_segments(full_metadata)
    print(f"[{condition}] Built {len(segments_by_condition[condition])} segment(s) across every lineage tree in frames 1-{max_frame}")

## Run: cell morphology

In [ ]:
cell_results = list(process_modality('cell', segments_by_condition))

## Run: nuclear morphology

In [ ]:
nuc_results = list(process_modality('nuc', segments_by_condition))

## Export plot data

Combines every (modality, group) result above into one CSV
(`plot_data.csv`, alongside the PNGs/SVGs above): one row per
individual distance, tagged by `modality`, `group`, and `series`
(`'sisters'`, `'cousins'`, or `'null'`), with `p_value` repeated on
every row of its own (modality, group, series != 'null') so every
figure's full distributions, and each relationship's Mann-Whitney U
result, can be redrawn from this one flat file.

In [ ]:
plot_data = pd.concat([
    pd.concat([
        pd.DataFrame({'modality': r['modality'], 'group': r['group'], 'series': 'sisters',
                      'distance': r['sister_distance'], 'p_value': r['sister_p_value']}),
        pd.DataFrame({'modality': r['modality'], 'group': r['group'], 'series': 'cousins',
                      'distance': r['cousin_distance'], 'p_value': r['cousin_p_value']}),
        pd.DataFrame({'modality': r['modality'], 'group': r['group'], 'series': 'null',
                      'distance': r['null_distance'], 'p_value': np.nan}),
    ])
    for r in (cell_results + nuc_results)
], ignore_index=True)

plot_data_path = os.path.join(PLOTS_DIR, 'plot_data.csv')
plot_data.to_csv(plot_data_path, index=False)
print(f"  Saved plot data ({len(plot_data)} rows) to {plot_data_path}")